In [1]:
import os
import numpy as np
import pandas as pd

print("Đã import thành công các thư viện pandas, numpy, os!")

Đã import thành công các thư viện pandas, numpy, os!


In [2]:
# Tự động định vị đường dẫn thư mục dataset dù chạy notebook ở root hay trong src/
if os.path.exists('dataset'):
    dataset_dir = 'dataset'
elif os.path.exists('../dataset'):
    dataset_dir = '../dataset'
elif os.path.exists('Dataset'):
    dataset_dir = 'Dataset'
elif os.path.exists('../Dataset'):
    dataset_dir = '../Dataset'
else:
    dataset_dir = '.'

sp500_path = os.path.join(dataset_dir, '1.0.0_sp500.csv')
vix_path = os.path.join(dataset_dir, '1.0.0_vixcls.csv')

# Đọc 2 file dữ liệu thô (phiên bản 1.0.0)
df_sp500 = pd.read_csv(sp500_path)
df_vix = pd.read_csv(vix_path)

print(f"• S&P 500 Dataset (1.0.0) : {df_sp500.shape[0]} dòng, {df_sp500.shape[1]} cột")
print(f"• VIX Dataset (1.0.0)     : {df_vix.shape[0]} dòng, {df_vix.shape[1]} cột")
display(df_sp500.head(3))
display(df_vix.head(3))

• S&P 500 Dataset : 9253 dòng, 7 cột
• VIX Dataset     : 9585 dòng, 2 cột


,Date,Open,High,Low,Close,Adj Close,Volume
0,1990-01-02,353.399994,359.690002,351.980011,359.690002,359.690002,162070000
1,1990-01-03,359.690002,360.589996,357.890015,358.760010,358.760010,192330000
2,1990-01-04,358.760010,358.760010,352.890015,355.670013,355.670013,177000000


,observation_date,VIXCLS
0,1990-01-02,17.24
1,1990-01-03,18.19
2,1990-01-04,19.22


In [3]:
# 1. Chuyển cột ngày của S&P 500 sang datetime chuẩn (YYYY-MM-DD)
df_sp500['Date'] = pd.to_datetime(df_sp500['Date'])

# 2. Đồng bộ cột ngày của VIX sang datetime và đổi tên thành 'Date'
df_vix['Date'] = pd.to_datetime(df_vix['observation_date'])
df_vix = df_vix.drop(columns=['observation_date'])

# Đảm bảo VIXCLS có kiểu số (float), ép các ký tự lạ hoặc '.' thành NaN
df_vix['VIXCLS'] = pd.to_numeric(df_vix['VIXCLS'], errors='coerce')

print("Đã chuẩn hóa định dạng Date cho cả 2 DataFrame:")
print(f"- S&P 500 Date range: {df_sp500['Date'].min().date()} ➔ {df_sp500['Date'].max().date()}")
print(f"- VIX Date range    : {df_vix['Date'].min().date()} ➔ {df_vix['Date'].max().date()}")

Đã chuẩn hóa định dạng Date cho cả 2 DataFrame:
- S&P 500 Date range: 1990-01-02 ➔ 2026-09-29
- VIX Date range    : 1990-01-02 ➔ 2026-09-28


In [4]:
# Thực hiện ghép nối dữ liệu theo các ngày giao dịch của chỉ số S&P 500 (how='left')
df_merged = pd.merge(df_sp500, df_vix, on='Date', how='left')

# Sắp xếp dữ liệu theo thứ tự thời gian tăng dần
df_merged = df_merged.sort_values('Date').reset_index(drop=True)

print(f"Kích thước sau khi Merge: {df_merged.shape[0]} phiên giao dịch, {df_merged.shape[1]} cột.")
print("\nDanh sách các cột trong dataset đã merge:")
print(list(df_merged.columns))

Kích thước sau khi Merge: 9253 phiên giao dịch, 8 cột.

Danh sách các cột trong dataset đã merge:
['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'VIXCLS']


In [5]:
print("Số lượng giá trị khuyết thiếu (NaN) trên từng cột:")
print(df_merged.isnull().sum())

# Vì VIX là chuỗi thời gian tài chính, nếu có ngày lễ/thiếu phiên giao dịch thì điền giá trị ngày liền trước (Forward Fill)
if df_merged['VIXCLS'].isnull().sum() > 0:
    print("\nTiến hành điền khuyết (ffill) cho các phiên thiếu chỉ số VIX...")
    df_merged['VIXCLS'] = df_merged['VIXCLS'].ffill()

print("\nKiểm tra lại sau khi xử lý NaN:")
print(f"Tổng số giá trị khuyết còn lại: {df_merged.isnull().sum().sum()}")

Số lượng giá trị khuyết thiếu (NaN) trên từng cột:
Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
VIXCLS       4
dtype: int64

Tiến hành điền khuyết (ffill) cho các phiên thiếu chỉ số VIX...

Kiểm tra lại sau khi xử lý NaN:
Tổng số giá trị khuyết còn lại: 0


In [6]:
# Đường dẫn lưu file
output_path = os.path.join(dataset_dir, '1.1.0_data_cleaning.csv')

# Lưu ra file CSV
df_merged.to_csv(output_path, index=False)
print(f"✅ ĐÃ LƯU THÀNH CÔNG TẠI: {output_path}")

print("\n=== 5 DÒNG ĐẦU TIÊN CỦA DATASET SAU KHI MERGE ===")
display(df_merged.head())

print("\n=== 5 DÒNG CUỐI CÙNG ===")
display(df_merged.tail())

✅ ĐÃ LƯU THÀNH CÔNG TẠI: .\sp500_vix_merged_1990_present.csv

=== 5 DÒNG ĐẦU TIÊN CỦA DATASET SAU KHI MERGE ===


,Date,Open,High,Low,Close,Adj Close,Volume,VIXCLS
0,1990-01-02,353.399994,359.690002,351.980011,359.690002,359.690002,162070000,17.24
1,1990-01-03,359.690002,360.589996,357.890015,358.760010,358.760010,192330000,18.19
2,1990-01-04,358.760010,358.760010,352.890015,355.670013,355.670013,177000000,19.22
3,1990-01-05,355.670013,355.670013,351.350006,352.200012,352.200012,158530000,20.11
4,1990-01-08,352.200012,354.239990,350.540009,353.790009,353.790009,140110000,20.26



=== 5 DÒNG CUỐI CÙNG ===


,Date,Open,High,Low,Close,Adj Close,Volume,VIXCLS
9248,2026-09-23,7761.939941,7761.939941,7694.890137,7706.029785,7706.029785,5110910000,15.18
9249,2026-09-24,7666.990234,7719.009766,7662.569824,7704.129883,7704.129883,5316390000,15.67
9250,2026-09-25,7709.859863,7752.069824,7693.080078,7743.410156,7743.410156,4499180000,14.87
9251,2026-09-28,7721.700195,7724.149902,7666.600098,7683.689941,7683.689941,5129710000,16.07
9252,2026-09-29,7699.600098,7699.600098,7655.549805,7657.819824,7657.819824,801095778,16.07
